In [ ]:
# from dl import authClient
# authClient.login('sbowes')

'No password or token supplied'

In [1]:
import pandas as pd
import numpy as np
from dl import queryClient
from astropy.table import Table, vstack
from astropy.coordinates import SkyCoord
from astropy import units as u
from collections import defaultdict

In [2]:
# ------------------------------
# User settings
# ------------------------------
input_file = 'summary_results15.csv'   # file with RA/DEC
output_file = 'smash_targets.fits'     # Combined output FITS
ra_col = 'RA'                           # RA column name
dec_col = 'DEC'                         # DEC column name
search_radius_deg = 0.0002778           # 1 arcsec radius
chunk_size = 200                        # Number of points per query chunk

# ------------------------------
# Load target table
# ------------------------------
targets = pd.read_csv(input_file)
# Check column names
print(targets.columns)

print(f"Loaded {len(targets)} targets from {input_file}")

# ------------------------------
# Split into chunks to avoid overly long queries
# ------------------------------
raw_chunks = np.array_split(targets, max(1, len(targets) // chunk_size + 1))
chunks = [pd.DataFrame(c, columns=targets.columns) for c in raw_chunks]  # Force columns

all_results = []
chunk_summary = []

for i, chunk in enumerate(chunks, 1):
    print(f"\nProcessing chunk {i}/{len(chunks)} with {len(chunk)} targets")
    
    # Build ADQL q3c query
    clauses = [
        f"q3c_radial_query(ra, dec, {row[ra_col]}, {row[dec_col]}, {search_radius_deg})"
        for _, row in chunk.iterrows()
    ]
    
    adql_query = f"""
    SELECT *
    FROM smash_dr2.object
    WHERE {" OR ".join(clauses)}
    """
    
    # Run query and save results
    result_file = f'chunk_{i}.fits'
    print("Submitting query...")
    queryClient.query(sql=adql_query, fmt='fits', out=result_file)
    
    # Check if any rows were returned
    t = Table.read(result_file)
    n_rows = len(t)
    chunk_summary.append((i, n_rows))
    
    if n_rows > 0:
        all_results.append(result_file)
        print(f"Chunk {i} returned {n_rows} rows and saved to {result_file}")
    else:
        print(f"Chunk {i} returned 0 rows, skipping")

# ------------------------------
# Combine non-empty FITS files
# ------------------------------
if all_results:
    tables = [Table.read(f) for f in all_results]
    combined_table = vstack(tables)
    combined_table.write(output_file, overwrite=True)
    print(f"\nAll chunks combined into {output_file}")
else:
    print("\nNo data returned from any chunk!")

# ------------------------------
# Summary
# ------------------------------
print("\nChunk summary (chunk number, rows returned):")
for i, n in chunk_summary:
    print(f"Chunk {i}: {n} rows")



Index(['RA', 'DEC', 'overall_mean', 'overall_median', 'overall_mean_mag',
       'overall_median_mag', 'offset_flag', 'chi_squared_68',
       'chi2_threshold_68', 'chi_flag_68', 'chi_squared_95',
       'chi2_threshold_95', 'chi_flag_95', 'chi_squared_997',
       'chi2_threshold_997', 'chi_flag_997', 'amplitude_5',
       'lower_percentile_5', 'upper_percentile_5', 'mag_amplitude_5',
       'amplitude_1', 'lower_percentile_1', 'upper_percentile_1',
       'mag_amplitude_1', 'largest_amp', 'best_period', 'alarm_level_flag',
       'std_amplitude', 'logT', 'logL', 'cephied_like', 'ir_variable_like',
       'other_variable_like'],
      dtype='str')
Loaded 848 targets from summary_results15.csv

Processing chunk 1/5 with 170 targets
Submitting query...
Chunk 1 returned 223 rows and saved to chunk_1.fits

Processing chunk 2/5 with 170 targets
Submitting query...
Chunk 2 returned 213 rows and saved to chunk_2.fits

Processing chunk 3/5 with 170 targets
Submitting query...
Chunk 3 returned

In [3]:
# Load the original targets and results
targets = pd.read_csv(input_file)
smash_results = Table.read(output_file)

print(f"Original targets: {len(targets)}")
print(f"SMASH DR2 results: {len(smash_results)}")
print(f"Expected 848, got {len(smash_results)} results")
print(f"Extra matches: {len(smash_results) - len(targets)}")

Original targets: 848
SMASH DR2 results: 1011
Expected 848, got 1011 results
Extra matches: 163


In [4]:
# Create SkyCoord objects for efficient matching
# Convert to numpy arrays and ensure they're float64
target_ra_vals = np.array(targets['RA'], dtype=np.float64)
target_dec_vals = np.array(targets['DEC'], dtype=np.float64)
smash_ra_vals = np.array(smash_results['ra'], dtype=np.float64)
smash_dec_vals = np.array(smash_results['dec'], dtype=np.float64)

target_coords = SkyCoord(ra=target_ra_vals*u.deg, dec=target_dec_vals*u.deg)
smash_coords = SkyCoord(ra=smash_ra_vals*u.deg, dec=smash_dec_vals*u.deg)

# For each SMASH result, find the closest target coordinate
idx_closest_targets, d2d, d3d = smash_coords.match_to_catalog_sky(target_coords)

# Convert distances to arcseconds
distances_arcsec = d2d.to(u.arcsec).value

print(f"Distance statistics (arcsec):")
print(f"Min: {distances_arcsec.min():.4f}")
print(f"Max: {distances_arcsec.max():.4f}")
print(f"Mean: {distances_arcsec.mean():.4f}")
print(f"Median: {np.median(distances_arcsec):.4f}")

Distance statistics (arcsec):
Min: 0.0021
Max: 1.0000
Mean: 0.2298
Median: 0.1141


In [5]:
# Analyze multiple matches for each target coordinate
target_match_counts = defaultdict(list)
for i, target_idx in enumerate(idx_closest_targets):
    target_match_counts[target_idx].append(i)

# Create summary statistics
match_counts = [len(matches) for matches in target_match_counts.values()]
unique_targets_matched = len(target_match_counts)
targets_with_multiple_matches = sum(1 for count in match_counts if count > 1)

print(f"\nMatching Summary:")
print(f"  Total target coordinates: {len(targets)}")
print(f"  Targets with at least one match: {unique_targets_matched}")
print(f"  Targets with multiple matches: {targets_with_multiple_matches}")
print(f"  Targets with no matches: {len(targets) - unique_targets_matched}")
print(f"  Total SMASH objects found: {len(smash_results)}")

print(f"\nMatch count distribution:")
from collections import Counter
count_distribution = Counter(match_counts)
for n_matches in sorted(count_distribution.keys()):
    n_targets = count_distribution[n_matches]
    print(f"  {n_targets} targets have {n_matches} match(es)")


Matching Summary:
  Total target coordinates: 848
  Targets with at least one match: 816
  Targets with multiple matches: 154
  Targets with no matches: 32
  Total SMASH objects found: 1011

Match count distribution:
  662 targets have 1 match(es)
  122 targets have 2 match(es)
  23 targets have 3 match(es)
  9 targets have 4 match(es)


In [6]:
# analysis of targets with multiple matches
multiple_match_details = []

for target_idx, smash_indices in target_match_counts.items():
    if len(smash_indices) > 1:
        target_ra = targets.iloc[target_idx]['RA']
        target_dec = targets.iloc[target_idx]['DEC']
        
        print(f"\nTarget {target_idx}: RA={target_ra:.6f}, DEC={target_dec:.6f}")
        print(f"  Found {len(smash_indices)} matches:")
        
        match_info = {
            'target_idx': target_idx,
            'target_ra': target_ra,
            'target_dec': target_dec,
            'n_matches': len(smash_indices),
            'smash_indices': smash_indices,
            'match_details': []
        }
        
        for i, smash_idx in enumerate(smash_indices):
            smash_ra = smash_results['ra'][smash_idx]
            smash_dec = smash_results['dec'][smash_idx]
            distance = distances_arcsec[smash_idx]
            
            print(f"    Match {i+1}: SMASH idx={smash_idx}, RA={smash_ra:.6f}, DEC={smash_dec:.6f}, dist={distance:.3f}\"")
            
            match_details = {
                'smash_idx': smash_idx,
                'smash_ra': smash_ra,
                'smash_dec': smash_dec,
                'distance_arcsec': distance
            }
            match_info['match_details'].append(match_details)
            
        multiple_match_details.append(match_info)

print(f"\nFound {len(multiple_match_details)} targets with multiple matches")


Target 5: RA=7.981742, DEC=-73.585571
  Found 4 matches:
    Match 1: SMASH idx=9, RA=7.981183, DEC=-73.585696, dist=0.725"
    Match 2: SMASH idx=10, RA=7.981785, DEC=-73.585581, dist=0.057"
    Match 3: SMASH idx=11, RA=7.981216, DEC=-73.585481, dist=0.626"
    Match 4: SMASH idx=12, RA=7.982138, DEC=-73.585431, dist=0.644"

Target 4: RA=7.980047, DEC=-73.578545
  Found 4 matches:
    Match 1: SMASH idx=13, RA=7.979233, DEC=-73.578518, dist=0.834"
    Match 2: SMASH idx=14, RA=7.980073, DEC=-73.578656, dist=0.400"
    Match 3: SMASH idx=15, RA=7.980068, DEC=-73.578550, dist=0.027"
    Match 4: SMASH idx=16, RA=7.980707, DEC=-73.578618, dist=0.722"

Target 2: RA=7.925977, DEC=-73.531670
  Found 2 matches:
    Match 1: SMASH idx=17, RA=7.926168, DEC=-73.531815, dist=0.557"
    Match 2: SMASH idx=18, RA=7.925979, DEC=-73.531678, dist=0.028"

Target 43: RA=11.938212, DEC=-73.307259
  Found 2 matches:
    Match 1: SMASH idx=39, RA=11.938252, DEC=-73.307434, dist=0.631"
    Match 2: SMASH

In [7]:
# Find missing targets (those with no matches)
matched_target_indices = set(target_match_counts.keys())
all_target_indices = set(range(len(targets)))
missing_target_indices = all_target_indices - matched_target_indices

print(f"Number of missing targets: {len(missing_target_indices)}")

if len(missing_target_indices) > 0:
    missing_details = []
    for target_idx in sorted(missing_target_indices):
        target_ra = targets.iloc[target_idx]['RA']
        target_dec = targets.iloc[target_idx]['DEC']
        
        missing_details.append({
            'target_idx': target_idx,
            'ra': target_ra,
            'dec': target_dec
        })
        
        print(f"  Target {target_idx}: RA={target_ra:.6f}, DEC={target_dec:.6f}")
    
    # Save missing targets to file for further investigation
    missing_df = pd.DataFrame(missing_details)
    missing_df.to_csv('missing_smash_targets.csv', index=False)
    print(f"\nMissing targets saved to 'missing_smash_targets.csv'")
else:
    print("Great! All targets have at least one match.")

Number of missing targets: 32
  Target 378: RA=69.807742, DEC=-71.735428
  Target 391: RA=72.301680, DEC=-69.456535
  Target 411: RA=73.033181, DEC=-66.818192
  Target 413: RA=73.073257, DEC=-66.912819
  Target 415: RA=73.222376, DEC=-67.095154
  Target 425: RA=73.566326, DEC=-66.301208
  Target 426: RA=73.573663, DEC=-67.093658
  Target 433: RA=73.738997, DEC=-66.752464
  Target 438: RA=73.839980, DEC=-67.436485
  Target 445: RA=74.063274, DEC=-66.203690
  Target 446: RA=74.073144, DEC=-66.305267
  Target 451: RA=74.127341, DEC=-66.302498
  Target 470: RA=74.437683, DEC=-65.708374
  Target 500: RA=75.247440, DEC=-66.644096
  Target 509: RA=75.744805, DEC=-65.937920
  Target 518: RA=76.041510, DEC=-67.330482
  Target 529: RA=76.336289, DEC=-70.741852
  Target 533: RA=76.474828, DEC=-68.180702
  Target 562: RA=77.286327, DEC=-68.985405
  Target 587: RA=78.378220, DEC=-69.539902
  Target 592: RA=78.507998, DEC=-67.451942
  Target 593: RA=78.518418, DEC=-67.264061
  Target 594: RA=78.5461

In [8]:
# Create a table that maps each original target to its SMASH matches
mapping_data = []

for target_idx in range(len(targets)):
    target_ra = targets.iloc[target_idx]['RA']
    target_dec = targets.iloc[target_idx]['DEC']
    
    if target_idx in target_match_counts:
        # Has matches
        smash_indices = target_match_counts[target_idx]
        n_matches = len(smash_indices)
        
        for i, smash_idx in enumerate(smash_indices):
            smash_ra = smash_results['ra'][smash_idx]
            smash_dec = smash_results['dec'][smash_idx]
            distance = distances_arcsec[smash_idx]
            
            mapping_data.append({
                'target_idx': target_idx,
                'target_ra': target_ra,
                'target_dec': target_dec,
                'n_matches_total': n_matches,
                'match_number': i + 1,
                'smash_idx': smash_idx,
                'smash_ra': smash_ra,
                'smash_dec': smash_dec,
                'distance_arcsec': distance,
                'has_match': True
            })
    else:
        # No matches
        mapping_data.append({
            'target_idx': target_idx,
            'target_ra': target_ra,
            'target_dec': target_dec,
            'n_matches_total': 0,
            'match_number': 0,
            'smash_idx': -1,
            'smash_ra': np.nan,
            'smash_dec': np.nan,
            'distance_arcsec': np.nan,
            'has_match': False
        })

# Convert to DataFrame and save
mapping_df = pd.DataFrame(mapping_data)
mapping_df.to_csv('target_smash_mapping.csv', index=False)

print(f"Created comprehensive mapping table with {len(mapping_df)} rows")
print(f"Saved to 'target_smash_mapping.csv'")
print(f"Columns: {list(mapping_df.columns)}")

# Quick summary of the mapping
print(f"\nMapping summary:")
print(f"  Rows with matches: {mapping_df['has_match'].sum()}")
print(f"  Rows without matches: {(~mapping_df['has_match']).sum()}")
print(f"  Unique targets: {mapping_df['target_idx'].nunique()}")
print(f"  Total SMASH objects: {mapping_df['smash_idx'][mapping_df['smash_idx'] >= 0].nunique()}")

Created comprehensive mapping table with 1043 rows
Saved to 'target_smash_mapping.csv'
Columns: ['target_idx', 'target_ra', 'target_dec', 'n_matches_total', 'match_number', 'smash_idx', 'smash_ra', 'smash_dec', 'distance_arcsec', 'has_match']

Mapping summary:
  Rows with matches: 1011
  Rows without matches: 32
  Unique targets: 848
  Total SMASH objects: 1011


# Creating cleaned SMASH data with brightest matches only

In [9]:
# Examine the SMASH data structure
print("SMASH DR2 data columns:")
print(smash_results.colnames)
print(f"\nTotal columns: {len(smash_results.colnames)}")

# Check for magnitude columns
mag_columns = [col for col in smash_results.colnames if 'mag' in col.lower() or col.lower() in ['u', 'g', 'r', 'i', 'z']]
print(f"\nPotential magnitude columns: {mag_columns}")

# Check specifically for u-band magnitude
u_columns = [col for col in smash_results.colnames if 'u' in col.lower() and 'mag' in col.lower()]
if not u_columns:
    u_columns = [col for col in smash_results.colnames if col.lower() == 'u']
print(f"\nU-band magnitude columns: {u_columns}")

# # Show a sample row to understand data types
# print(f"\nSample SMASH data (first row):")
# sample_row = smash_results[0]
# for col in smash_results.colnames[:10]:  # Show first 10 columns
#     print(f"  {col}: {sample_row[col]}")
# if len(smash_results.colnames) > 10:
#     print(f"  ... and {len(smash_results.colnames) - 10} more columns")

SMASH DR2 data columns:
['ra', 'dec', 'glon', 'glat', 'elon', 'elat', 'raerr', 'decerr', 'rascatter', 'decscatter', 'umag', 'uerr', 'uscatter', 'gmag', 'gerr', 'gscatter', 'rmag', 'rerr', 'rscatter', 'imag', 'ierr', 'iscatter', 'zmag', 'zerr', 'zscatter', 'chi', 'sharp', 'prob', 'ebv', 'htm9', 'ring256', 'nest4096', 'random_id', 'ndet', 'depthflag', 'ndetu', 'ndetg', 'ndetr', 'ndeti', 'ndetz', 'flag', 'id']

Total columns: 42

Potential magnitude columns: ['umag', 'gmag', 'rmag', 'imag', 'zmag']

U-band magnitude columns: ['umag']


In [10]:
# Create the cleaned dataset with only the brightest U-band matches
print("Creating cleaned SMASH dataset...")
print("=" * 50)

# Initialize the final dataset list
cleaned_data = []

# Process each target
for target_idx in range(len(targets)):
    # Get target information
    target_ra = targets.iloc[target_idx]['RA']
    target_dec = targets.iloc[target_idx]['DEC']
    
    # Create base row with target coordinates and other target data
    row_data = {
        'target_idx': target_idx,
        'target_ra': target_ra,
        'target_dec': target_dec
    }
    
    # Add all other columns from the original target data
    for col in targets.columns:
        if col not in ['RA', 'DEC']:  # Avoid duplicates
            row_data[f'target_{col.lower()}'] = targets.iloc[target_idx][col]
    
    # Check if this target has matches
    if target_idx in target_match_counts:
        smash_indices = target_match_counts[target_idx]
        n_matches = len(smash_indices)
        
        # Set the multiple match flag
        multiple_match_flag = 1 if n_matches > 1 else 0
        row_data['multiple_match_flag'] = multiple_match_flag
        
        if n_matches == 1:
            # Single match - use it directly
            smash_idx = smash_indices[0]
            best_smash_row = smash_results[smash_idx]
        else:
            # Multiple matches - find the brightest (smallest umag)
            best_smash_idx = None
            best_umag = float('inf')
            
            for smash_idx in smash_indices:
                umag = smash_results['umag'][smash_idx]
                if not np.isnan(umag) and umag < best_umag:
                    best_umag = umag
                    best_smash_idx = smash_idx
            
            # If all have NaN umag, just take the first one
            if best_smash_idx is None:
                best_smash_idx = smash_indices[0]
            
            best_smash_row = smash_results[best_smash_idx]
        
        # Add SMASH data to the row
        for col in smash_results.colnames:
            row_data[f'smash_{col}'] = best_smash_row[col]
        
        # Add the distance information
        smash_idx_in_array = np.where(idx_closest_targets == target_idx)[0]
        if len(smash_idx_in_array) > 0:
            # Find the distance for this specific match
            matching_distances = []
            for i, closest_target in enumerate(idx_closest_targets):
                if closest_target == target_idx:
                    matching_distances.append(distances_arcsec[i])
            
            if len(matching_distances) > 0:
                # For multiple matches, we'll report the distance of the brightest match
                if n_matches == 1:
                    row_data['distance_arcsec'] = matching_distances[0]
                else:
                    # Find which distance corresponds to our chosen brightest match
                    for i, smash_idx in enumerate(smash_indices):
                        if smash_idx == best_smash_idx:
                            idx_in_distances = 0
                            for j, closest_target in enumerate(idx_closest_targets):
                                if closest_target == target_idx:
                                    if idx_in_distances == i:
                                        row_data['distance_arcsec'] = distances_arcsec[j]
                                        break
                                    idx_in_distances += 1
                            break
            else:
                row_data['distance_arcsec'] = np.nan
        else:
            row_data['distance_arcsec'] = np.nan
            
    else:
        # No matches found
        row_data['multiple_match_flag'] = 0
        
        # Add NaN values for all SMASH columns
        for col in smash_results.colnames:
            row_data[f'smash_{col}'] = np.nan
            
        row_data['distance_arcsec'] = np.nan
    
    cleaned_data.append(row_data)

print(f"Processed {len(cleaned_data)} targets")

# Convert to DataFrame for easier handling
cleaned_df = pd.DataFrame(cleaned_data)
print(f"Created DataFrame with shape: {cleaned_df.shape}")
print(f"Columns: {list(cleaned_df.columns)}")

# Check the multiple match flag distribution
flag_counts = cleaned_df['multiple_match_flag'].value_counts().sort_index()
print(f"\nMultiple match flag distribution:")
print(f"  0 (single/no matches): {flag_counts.get(0, 0)}")
print(f"  1 (multiple matches): {flag_counts.get(1, 0)}")

Creating cleaned SMASH dataset...
Processed 848 targets
Created DataFrame with shape: (848, 78)
Columns: ['target_idx', 'target_ra', 'target_dec', 'target_overall_mean', 'target_overall_median', 'target_overall_mean_mag', 'target_overall_median_mag', 'target_offset_flag', 'target_chi_squared_68', 'target_chi2_threshold_68', 'target_chi_flag_68', 'target_chi_squared_95', 'target_chi2_threshold_95', 'target_chi_flag_95', 'target_chi_squared_997', 'target_chi2_threshold_997', 'target_chi_flag_997', 'target_amplitude_5', 'target_lower_percentile_5', 'target_upper_percentile_5', 'target_mag_amplitude_5', 'target_amplitude_1', 'target_lower_percentile_1', 'target_upper_percentile_1', 'target_mag_amplitude_1', 'target_largest_amp', 'target_best_period', 'target_alarm_level_flag', 'target_std_amplitude', 'target_logt', 'target_logl', 'target_cephied_like', 'target_ir_variable_like', 'target_other_variable_like', 'multiple_match_flag', 'smash_ra', 'smash_dec', 'smash_glon', 'smash_glat', 'smash

In [11]:
# Save the cleaned dataset as CSV
output_csv_file = 'smash_targets_cleaned.csv'
cleaned_df.to_csv(output_csv_file, index=False)

print(f"Cleaned dataset saved as: {output_csv_file}")
print(f"Shape: {cleaned_df.shape}")
print(f"Columns: {len(cleaned_df.columns)}")

# Quick summary of what we created
print(f"\nSummary of cleaned dataset:")
print(f"total targets: {len(cleaned_df)}")
print(f"targets with SMASH matches: {(~cleaned_df['smash_ra'].isna()).sum()}")
print(f"targets without matches: {cleaned_df['smash_ra'].isna().sum()}")
print(f"targets with multiple matches (flag=1): {(cleaned_df['multiple_match_flag'] == 1).sum()}")
print(f"for multiple matches: selected brightest U-band magnitude")
print(f"all target data preserved with 'target_' prefix")
print(f"all SMASH data included with 'smash_' prefix")
print(f"distance in arcseconds included")
print(f"multiple match flag: 0 = single/no match, 1 = multiple matches")

Cleaned dataset saved as: smash_targets_cleaned.csv
Shape: (848, 78)
Columns: 78

Summary of cleaned dataset:
total targets: 848
targets with SMASH matches: 816
targets without matches: 32
targets with multiple matches (flag=1): 154
for multiple matches: selected brightest U-band magnitude
all target data preserved with 'target_' prefix
all SMASH data included with 'smash_' prefix
distance in arcseconds included
multiple match flag: 0 = single/no match, 1 = multiple matches


# Corrected Brightest U-band Selection

In [15]:
# Add SMASH U magnitude and error to Anna's candidate files

# Load the cleaned SMASH data
if 'cleaned_df' not in globals():
    cleaned_df = pd.read_csv('smash_targets_cleaned.csv')

print(f"Loaded cleaned SMASH data with {len(cleaned_df)} targets")

candidate_files = [
    './annas_candidates/final_lmc_ysgcands.csv',
    './annas_candidates/final_smc_ysgcands.csv'
]

# Function to match coordinates with tolerance
def find_closest_smash_match(ra, dec, smash_data, tolerance_arcsec=1.0):
    """Find the closest SMASH match within tolerance"""
    # Convert tolerance from arcseconds to degrees
    tolerance_deg = tolerance_arcsec / 3600.0
    
    # Calculate distances
    distances = np.sqrt((smash_data['target_ra'] - ra)**2 + 
                       (smash_data['target_dec'] - dec)**2)
    
    # Find closest match within tolerance
    closest_idx = np.argmin(distances)
    closest_distance = distances.iloc[closest_idx]
    
    if closest_distance <= tolerance_deg:
        return smash_data.iloc[closest_idx]
    else:
        return None

# Process each candidate file
for file_path in candidate_files:
    print(f"\nProcessing: {file_path}")
    
    try:
        # Load the candidate file
        candidates = pd.read_csv(file_path)
        print(f"  Loaded {len(candidates)} candidates")
        print(f"  Original columns: {list(candidates.columns)}")
        
        # Identify RA/DEC columns (try common column names)
        ra_col = None
        dec_col = None
        
        # Check for common RA/DEC column names
        for col in candidates.columns:
            if col.upper() in ['RA', '_RA', 'RA_DEG', 'RADEG']:
                ra_col = col
            elif col.upper() in ['DEC', '_DEC', 'DEC_DEG', 'DECDEG', 'DECL']:
                dec_col = col
        
        # If not found, try partial matches
        if ra_col is None:
            ra_candidates = [col for col in candidates.columns if 'ra' in col.lower()]
            if ra_candidates:
                ra_col = ra_candidates[0]
                
        if dec_col is None:
            dec_candidates = [col for col in candidates.columns if 'dec' in col.lower()]
            if dec_candidates:
                dec_col = dec_candidates[0]
        
        if ra_col is None or dec_col is None:
            print(f"  ERROR: Could not identify RA/DEC columns in {file_path}")
            print(f"  Available columns: {list(candidates.columns)}")
            continue
            
        print(f"  Using RA column: {ra_col}")
        print(f"  Using DEC column: {dec_col}")
        
        # Initialize new columns with NaN
        candidates['Usmashmag'] = np.nan
        candidates['e_Usmash'] = np.nan
        
        # Match coordinates and add U magnitude data
        matches_found = 0
        for idx, row in candidates.iterrows():
            ra = row[ra_col]
            dec = row[dec_col]
            
            # Find closest SMASH match
            smash_match = find_closest_smash_match(ra, dec, cleaned_df)
            
            if smash_match is not None:
                # Check if SMASH data has valid U magnitude
                if not pd.isna(smash_match['smash_umag']):
                    candidates.at[idx, 'Usmashmag'] = smash_match['smash_umag']
                
                if not pd.isna(smash_match['smash_uerr']):
                    candidates.at[idx, 'e_Usmash'] = smash_match['smash_uerr']
                
                matches_found += 1
        
        print(f"Found SMASH matches for {matches_found}/{len(candidates)} candidates")
        
        # Count how many got U magnitude data
        with_umag = (~candidates['Usmashmag'].isna()).sum()
        with_uerr = (~candidates['e_Usmash'].isna()).sum()
        print(f"Added U magnitude to {with_umag} candidates")
        print(f"Added U magnitude error to {with_uerr} candidates")
        
        # Save the updated file
        candidates.to_csv(file_path, index=False)
        print(f"Updated file saved: {file_path}")
        print(f"New columns: {list(candidates.columns)}")
        
    except FileNotFoundError:
        print(f"  ERROR: File not found: {file_path}")
    except Exception as e:
        print(f"  ERROR processing {file_path}: {e}")

print(f"\n{'='*60}")
print("COMPLETED: Added SMASH U magnitudes to candidate files")
print("New columns added: 'Usmashmag', 'e_Usmash'")

Loaded cleaned SMASH data with 848 targets

Processing: ./annas_candidates/final_lmc_ysgcands.csv
  Loaded 471 candidates
  Original columns: ['ra', 'dec', '2MASS', 'ra_gaia', 'dec_gaia', 'parallax', 'pmra', 'pmdec', 'pm', 'ra_error', 'dec_error', 'parallax_error', 'parallax_over_error', 'pmra_error', 'pmdec_error', 'astrometric_gof_al', 'astrometric_chi2_al', 'astrometric_excess_noise', 'astrometric_excess_noise_sig', 'ruwe', 'astrometric_params_solved', 'astrometric_sigma5d_max', 'duplicated_source', 'phot_g_mean_mag', 'phot_bp_mean_mag', 'phot_rp_mean_mag', 'phot_g_mean_flux', 'phot_bp_mean_flux', 'phot_rp_mean_flux', 'phot_g_mean_flux_error', 'phot_g_mean_flux_over_error', 'phot_bp_mean_flux_error', 'phot_bp_mean_flux_over_error', 'phot_rp_mean_flux_error', 'phot_rp_mean_flux_over_error', 'phot_bp_rp_excess_factor', 'Separation_1', 'covariance', 'Jmag', 'Hmag', 'Kmag', 'e_Jmag', 'e_Hmag', 'e_Kmag', 'Qfl', 'Umag', 'e_Umag', 'Bmag', 'e_Bmag', 'Vmag', 'e_Vmag', 'Imag', 'e_Imag', 'uvw1

/opt/anaconda3/envs/datalab/lib/python3.14/site-packages/pandas/core/internals/managers.py:2209: UserWarning: Warning: converting a masked element to nan.
  arr[indexer] = value
